# Munir (منير) — Run All

**SDAIA Academy Capstone — SDA-AIE-213: LLM Application Engineering**

This notebook is the single entry point for running and evaluating the Munir application.

### Default backend

Munir uses the deterministic mock backend by default, so no API key is required.

### Run

Use **Kernel → Restart Kernel and Run All** to execute the complete evaluation.

The notebook runs:

1. Automated tests
2. Golden-set evaluation
3. Judge calibration
4. Regression gate
5. Cost and latency replay
6. Commercial vs open-weight comparison
7. Self-host break-even analysis

The generated evaluation artifacts are written to the repository's `eval/out/` directory.

In [ ]:
# ============================================================
# 2. Environment setup
# ============================================================

from pathlib import Path
import os
import subprocess
import sys

# Detect whether the notebook is running in Google Colab.
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Environment: Google Colab")

    # Clone the repository only if it is not already present.
    REPO_DIR = Path("/content/munir")

    if not REPO_DIR.exists():
        REPO_URL = "https://github.com/Lobaali/munir.git"

        subprocess.run(
            ["git", "clone", REPO_URL, str(REPO_DIR)],
            check=True,
        )

else:
    print("Environment: Local Jupyter / VS Code")

    # When running locally, use the repository containing this notebook.
    REPO_DIR = Path.cwd()

# Move into the repository so every later command uses the same root.
os.chdir(REPO_DIR)

# Make the source package importable.
src_dir = REPO_DIR / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f"Repository root: {REPO_DIR}")
print(f"Python executable: {sys.executable}")
print("✓ Environment configured.")

Repository: /Users/loba/Munir/munir
Python: /usr/local/bin/python3.12
PYTHONPATH: /Users/loba/Munir/munir:/Users/loba/Munir/munir/src:/Users/loba/Munir/munir:/Users/loba/Munir/munir/src:/Users/loba/Munir/munir:/Users/loba/Munir/munir/src


In [ ]:
# ============================================================
# 2. Verify project structure
# =================================xq==========================

required_paths = [
    Path("src/munir"),
    Path("configs/munir.yaml"),
    Path("data"),
    Path("eval"),
    Path("scripts"),
    Path("tests"),
    Path("BENCHMARKS.md"),
    Path("EVALUATION_REPORT.md"),
    Path("DECISIONS.md"),
]

missing = [str(path) for path in required_paths if not path.exists()]

if missing:
    raise FileNotFoundError(
        "The following required project paths are missing:\n"
        + "\n".join(f"- {path}" for path in missing)
    )

print("✓ Munir project structure verified.")

Environment: Local Jupyter / VS Code
Repository root: /Users/loba/Munir/munir
Python executable: /usr/local/bin/python3.12
✓ Environment configured.


In [33]:
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
requirements = ROOT / "requirements.txt"

if not requirements.exists():
    raise FileNotFoundError(f"Could not find {requirements}")

print(f"Installing project dependencies from: {requirements}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements)],
    check=True,
)

print("\nRunning automated test suite...\n")

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    check=False,
    capture_output=True,
    text=True,
)

print(result.stdout)

if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "The automated test suite failed. "
        "Review the pytest output above before continuing."
    )

print("\n✓ Automated test suite passed.")

Installing project dependencies from: /Users/loba/Munir/munir/requirements.txt



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip



Running automated test suite...

......................................                                   [100%]
38 passed in 0.34s


✓ Automated test suite passed.


In [34]:
# ============================================================
# 4. Run the evaluation pipeline
# ============================================================

commands = [
    (
        "Golden-set evaluation",
        [sys.executable, "eval/harness.py"],
    ),
    (
        "Judge calibration",
        [sys.executable, "eval/calibrate_judge.py"],
    ),
]

for name, command in commands:
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    result = subprocess.run(
        command,
        check=False,
        capture_output=True,
        text=True,
    )

    if result.stdout:
        print(result.stdout)

    if result.stderr:
        print("\n--- STDERR ---")
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(
            f"{name} failed with exit code {result.returncode}."
        )


# ------------------------------------------------------------
# Regression gate
# The harness above generates eval/out/eval_run.json.
# We pass that freshly generated report to the gate.
# ------------------------------------------------------------

report_path = Path("eval/out/eval_run.json")

if not report_path.exists():
    raise FileNotFoundError(
        f"Expected evaluation report was not generated: {report_path}"
    )

print("\n" + "=" * 70)
print("Regression gate")
print("=" * 70)

result = subprocess.run(
    [
        sys.executable,
        "eval/gate.py",
        str(report_path),
    ],
    check=False,
    capture_output=True,
    text=True,
)

if result.stdout:
    print(result.stdout)

if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f"Regression gate failed with exit code {result.returncode}."
    )

print("\n✓ Evaluation pipeline completed successfully.")


Golden-set evaluation

----------------------------------------------------------------------------
eval | route=default | 120 cases | pass 120/120 (100%) | 0.09s
----------------------------------------------------------------------------
  language    ar 100% | en 100%
  intent      escalate 100% | faq 100% | service 100%
  difficulty  adversarial 100% | edge 100% | routine 100%
  risk        normal 100% | safety 100%
  p50 latency 0.8 ms

  written: /Users/loba/Munir/munir/eval/out/eval_run.json


--- STDERR ---
2026-09-15T11:47:56.631317Z [info     ] structured_extracted           attempt=1 outcome=first_try schema=guard_classification trace_id=96a91ff6d693
2026-09-15T11:47:56.631607Z [info     ] llm_cost                       cache_tier= cached_tokens=0 cost_sar=0.00297 input_tokens=178 intent=unknown latency_ms=0.0 model_id=campus-flagship output_tokens=4 prompt_version= route= stage=input_guard trace_id=96a91ff6d693
2026-09-15T11:47:56.631853Z [info     ] structured_extracted  

In [35]:
# ============================================================
# 5. Run cost, cache, model-comparison, and break-even evidence
# ============================================================

commands = [
    (
        "Cost and latency replay",
        [
            sys.executable,
            "scripts/replay.py",
            "--limit",
            "120",
            "--write",
        ],
    ),
    (
        "Commercial vs open-weight comparison",
        [
            sys.executable,
            "scripts/compare_models.py",
            "--limit",
            "120",
        ],
    ),
    (
        "Self-host break-even analysis",
        [
            sys.executable,
            "scripts/breakeven.py",
            "--gpu-usd-per-hour",
            "3.33",
            "--tokens-per-sec",
            "950",
            "--utilization",
            "0.50",
            "--commercial-price-per-mtok",
            "15",
            "--avg-tokens-per-request",
            "100",
        ],
    ),
]

for name, command in commands:
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    result = subprocess.run(
        command,
        check=False,
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"{name} failed with exit code {result.returncode}."
        )

print("\n✓ Cost, model comparison, and break-even analysis completed.")


Cost and latency replay


2026-09-15T11:47:57.487328Z [info     ] structured_extracted           attempt=1 outcome=first_try schema=guard_classification trace_id=c37d0d737c0f
2026-09-15T11:47:57.488394Z [info     ] llm_cost                       cache_tier= cached_tokens=0 cost_sar=0.00291 input_tokens=174 intent=unknown latency_ms=0.0 model_id=campus-flagship output_tokens=4 prompt_version= route= stage=input_guard trace_id=c37d0d737c0f
2026-09-15T11:47:57.488748Z [warning  ] structured_validation_failed   attempt=1 errors=[[]] schema=route_verdict trace_id=c37d0d737c0f
2026-09-15T11:47:57.488826Z [info     ] structured_extracted           attempt=2 outcome=after_repair schema=route_verdict trace_id=c37d0d737c0f
2026-09-15T11:47:57.488866Z [info     ] llm_cost                       cache_tier= cached_tokens=0 cost_sar=0.00219 input_tokens=126 intent=unknown latency_ms=0.0 model_id=campus-flagship output_tokens=4 prompt_version= route= stage=router trace_id=c37d0d737c0f
2026-09-15T11:47:57.488902Z [info     ] l


Module 6 optimisation replay
step           cost SAR   cache input     p50 ms  model calls       eval     safety
--------------------------------------------------------------------------------------------------------------
before         0.189458        70.8%        0.9           35    100.0%    100.0%
prompt         0.189458        70.8%        0.7           35    100.0%    100.0%
cache          0.117870        64.3%        0.6           31    100.0%    100.0%
cascade        0.117870        64.3%        0.8           31    100.0%    100.0%

The evaluation columns are the release verdict, not decoration.
If cost falls but the safety slice falls below 100%, do not ship the step.

written: /Users/loba/Munir/munir/eval/out/cost_optimization_comparison.json

Commercial vs open-weight comparison


2026-09-15T11:48:00.036360Z [info     ] structured_extracted           attempt=1 outcome=first_try schema=guard_classification trace_id=a2390e86ef3a
2026-09-15T11:48:00.036855Z [info     ] llm_cost                       cache_tier= cached_tokens=0 cost_sar=0.00297 input_tokens=178 intent=unknown latency_ms=0.0 model_id=campus-flagship output_tokens=4 prompt_version= route= stage=input_guard trace_id=a2390e86ef3a
2026-09-15T11:48:00.037360Z [info     ] structured_extracted           attempt=1 outcome=first_try schema=route_verdict trace_id=a2390e86ef3a
2026-09-15T11:48:00.037447Z [info     ] llm_cost                       cache_tier= cached_tokens=0 cost_sar=0.0021 input_tokens=130 intent=unknown latency_ms=0.0 model_id=campus-flagship output_tokens=2 prompt_version= route= stage=router trace_id=a2390e86ef3a
2026-09-15T11:48:00.037524Z [info     ] routed                         intent=faq prompt_version=route_intent.v1 trace_id=a2390e86ef3a
2026-09-15T11:48:00.047883Z [info     ] llm_co


COMMERCIAL VS OPEN-WEIGHT
primary      pass=119/120 (99%) | cost=1.6338 SAR | calls=335 | benchmark tok/s=950.00
  observed tok/s: 0.00
  language   : ar=100% | en=99%
  intent     : escalate=100% | faq=100% | service=97%
  difficulty : adversarial=100% | edge=100% | routine=98%
  risk       : normal=99% | safety=100%
open_weight  pass=117/120 (98%) | cost=0.1686 SAR | calls=335 | benchmark tok/s=950.00
  observed tok/s: 0.00
  language   : ar=98% | en=97%
  intent     : escalate=100% | faq=97% | service=97%
  difficulty : adversarial=100% | edge=93% | routine=98%
  risk       : normal=97% | safety=100%

SLICE DELTAS (second route minus first route)
  language    ar           -2%
  language    en           -1%
  intent      escalate     +0%
  intent      faq          -3%
  intent      service      +0%
  difficulty  adversarial  +0%
  difficulty  edge         -7%
  difficulty  routine      +0%
  risk        normal       -2%
  risk        safety       +0%

written: /Users/loba/Munir/mun

In [36]:
# ============================================================
# 6. Verify generated evaluation artifacts
# ============================================================

print("=" * 70)
print("GENERATED EVIDENCE")
print("=" * 70)

files_to_check = [
    Path("BENCHMARKS.md"),
    Path("EVALUATION_REPORT.md"),
    Path("DECISIONS.md"),
    Path("eval/out/cost_optimization_comparison.json"),
    Path("eval/out/commercial_vs_open_weight.json"),
]

for path in files_to_check:
    if path.exists():
        print(f"✓ {path}")
    else:
        print(f"⚠ Missing: {path}")

print("\nRun All completed.")

GENERATED EVIDENCE
✓ BENCHMARKS.md
✓ EVALUATION_REPORT.md
✓ DECISIONS.md
✓ eval/out/cost_optimization_comparison.json
✓ eval/out/commercial_vs_open_weight.json

Run All completed.
